In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import re
import os
from tqdm import tqdm
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

D = 256
H = 4
N = 32768
VOCAB_SIZE = 50257 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"System Ready. Using Device: {DEVICE}")


def find_file(filename_part):
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if filename_part in name:
                return os.path.join(root, name)
    if os.path.exists(filename_part):
        return filename_part
    print(f"CRITICAL WARNING: File containing '{filename_part}' not found!")
    return None

train_path = find_file('train.csv')
novel_1 = find_file('The Count of Monte Cristo.txt')
novel_2 = find_file('In search of the castaways.txt')

novel_files = [f for f in [novel_1, novel_2] if f is not None]

print(f"Training Data: {train_path}")
print(f"Novels Found: {novel_files}")
class BDH_GPU(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(VOCAB_SIZE, D)
        self.ln = nn.LayerNorm(D)
        self.decoder_y = nn.Parameter(torch.randn(H, D, N // H) * 0.02)
        
    def forward(self, idx):
        x = self.wte(idx)
        x = self.ln(x)
        k = torch.einsum('btd,hdn->bhtn', x, self.decoder_y)
        k = F.relu(k)
        return x
class CausalMemory(nn.Module):
    def __init__(self):
        super().__init__()
        self.state = nn.Parameter(torch.randn(N, D) * 0.01)
        self.time_proj = nn.Linear(1, D)
        self.loc_proj = nn.Linear(1, D)
        self.gate_net = nn.Sequential(
            nn.Linear(D * 2, D), nn.ReLU(),
            nn.Linear(D, 1), nn.Sigmoid()
        )
        self.forget_net = nn.Sequential(
            nn.Linear(D * 2, 1), nn.Sigmoid()
        )
        self.reconstruction_head = nn.Linear(D, D)

    def forward(self, e_id, attr_vec, t, l, current_memory=None):
        if current_memory is None:
            current_memory = self.state[e_id].unsqueeze(0)
        t_emb = torch.tanh(self.time_proj(torch.tensor([[float(t)]], device=DEVICE)))
        l_emb = torch.tanh(self.loc_proj(torch.tensor([[float(l)]], device=DEVICE)))
        contextual_fact = attr_vec * (t_emb + l_emb)
        combined = torch.cat([contextual_fact, current_memory], dim=-1)
        i_gate = self.gate_net(combined)
        f_gate = self.forget_net(combined)
        new_memory = (f_gate * current_memory) + (i_gate * contextual_fact)
        reconstructed = self.reconstruction_head(new_memory)
        
        return new_memory, i_gate, f_gate, reconstructed

def train_model_with_save(file_paths, train_df, encoder, memory, optimizer, epochs=5):
    encoder.train()
    memory.train()
    
    # Loss Weights
    W_CONSISTENCY = 2.0 
    W_RECON = 1.0       
    W_GATING = 0.1      
    W_DECAY = 0.01      
    
    best_f1 = -1.0
    
    print(f"Starting Training for {epochs} Epochs...")
    
    for epoch in range(epochs):
        novel_iterator = tqdm(file_paths, desc=f"Epoch {epoch+1}")
        
        for novel_path in novel_iterator:
            with open(novel_path, 'r', encoding='utf-8') as f: text = f.read()
            chapters = re.split(r'(?im)^\s*Chapter\s+(?:[0-9]+|[IVXLCDM]+)\b', text)
            chapters = [c for c in chapters if len(c) > 500]
            
            batch_loss_accum = 0
            for l_idx, chapter in enumerate(chapters[:10]): 
                words = chapter.split()[:150]
                for t_idx, word in enumerate(words):
                    optimizer.zero_grad()
                    
                    token_idx = torch.tensor([[hash(word) % VOCAB_SIZE]], device=DEVICE)
                    attr_vec = encoder(token_idx).squeeze(1)
                    e_id = hash(word) % N
                    
                    old_mem = memory.state[e_id].unsqueeze(0).clone()
                    new_mem, i_gate, f_gate, recon_vec = memory(e_id, attr_vec, t_idx, l_idx, old_mem)
                    memory.state.data[e_id] = new_mem.detach()
                    
                    loss_recon = F.mse_loss(recon_vec, attr_vec.detach())
                    loss_gating = torch.mean(torch.abs(i_gate))
                    loss_decay = torch.mean(torch.abs(1.0 - f_gate))
                    
                    loss_ingest = (loss_recon * W_RECON) + (loss_gating * W_GATING) + (loss_decay * W_DECAY)
                    loss_ingest.backward()
                    optimizer.step()
                    batch_loss_accum += loss_ingest.item()
            optimizer.zero_grad()
            batch_consistency_loss = 0
            sample_df = train_df.sample(min(32, len(train_df)))
            
            for _, row in sample_df.iterrows():
                b_words = str(row['content']).split()
                if not b_words: continue
                b_vecs = [encoder(torch.tensor([[hash(w)%VOCAB_SIZE]], device=DEVICE)).squeeze(1) for w in b_words]
                b_emb = torch.mean(torch.stack(b_vecs), dim=0)
                
                char_id = hash(row['char']) % N
                mem_trace = memory.reconstruction_head(memory.state[char_id].unsqueeze(0))
                
                sim = F.cosine_similarity(mem_trace, b_emb)
                target = 1.0 if row['label'] == 'consistent' else -1.0
                loss = F.relu(0.5 - sim) if target == 1 else F.relu(sim + 0.5)
                batch_consistency_loss += loss
            
            final_loss = batch_consistency_loss * W_CONSISTENCY
            final_loss.backward()
            optimizer.step()
            
            novel_iterator.set_postfix(ingest=batch_loss_accum/100, cons=final_loss.item())
        optimizer.zero_grad()
        val_df = train_df.sample(min(50, len(train_df))) 
        scores, labels = [], []
        label_map = {'consistent': 1, 'contradict': 0}
        
        with torch.no_grad():
            for _, row in val_df.iterrows():
                b_words = str(row['content']).split()
                if not b_words: continue
                b_vecs = [encoder(torch.tensor([[hash(w)%VOCAB_SIZE]], device=DEVICE)).squeeze(1) for w in b_words]
                b_emb = torch.mean(torch.stack(b_vecs), dim=0)
                char_id = hash(row['char']) % N
                mem_trace = memory.reconstruction_head(memory.state[char_id].unsqueeze(0))
                scores.append(F.cosine_similarity(mem_trace, b_emb).item())
                labels.append(label_map[row['label']])
        best_epoch_f1 = 0
        for t in np.linspace(-1, 1, 50):
            preds = [1 if s >= t else 0 for s in scores]
            curr_f1 = f1_score(labels, preds, average='weighted')
            if curr_f1 > best_epoch_f1: best_epoch_f1 = curr_f1
        
        print(f"Epoch {epoch+1} Val F1: {best_epoch_f1:.4f}")
        if best_epoch_f1 > best_f1:
            best_f1 = best_epoch_f1
            print(f"New Best Model (F1: {best_f1:.4f}). Saving...")
            torch.save({
                'encoder': encoder.state_dict(),
                'memory': memory.state_dict(),
                'optimizer': optimizer.state_dict(),
                'best_f1': best_f1
            }, MODEL_SAVE_PATH)

def generate_submission(train_df, test_file_path, encoder, memory):
    encoder.eval()
    memory.eval()
    print("Calibrating Threshold on Training Data...")
    scores, labels = [], []
    label_map = {'consistent': 1, 'contradict': 0}
    
    with torch.no_grad():
        for _, row in train_df.iterrows():
            b_words = str(row['content']).split()
            if not b_words: continue
            b_vecs = [encoder(torch.tensor([[hash(w)%VOCAB_SIZE]], device=DEVICE)).squeeze(1) for w in b_words]
            b_emb = torch.mean(torch.stack(b_vecs), dim=0)
            char_id = hash(row['char']) % N
            mem_trace = memory.reconstruction_head(memory.state[char_id].unsqueeze(0))
            scores.append(F.cosine_similarity(mem_trace, b_emb).item())
            labels.append(label_map[row['label']])
            
    best_thresh, best_acc = 0, 0
    for t in np.linspace(-1, 1, 100):
        preds = [1 if s >= t else 0 for s in scores]
        acc = balanced_accuracy_score(labels, preds)
        if acc > best_acc: best_acc, best_thresh = acc, t
    print(f"Optimal Threshold: {best_thresh:.4f}")
    print(f"Generating predictions for {test_file_path}...")
    test_df = pd.read_csv(test_file_path)
    final_ids = []
    final_preds = []
    
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
            b_words = str(row['content']).split()
            if not b_words:
                final_preds.append("consistent")
                final_ids.append(row['id'])
                continue
            b_vecs = [encoder(torch.tensor([[hash(w)%VOCAB_SIZE]], device=DEVICE)).squeeze(1) for w in b_words]
            b_emb = torch.mean(torch.stack(b_vecs), dim=0)
            char_id = hash(row['char']) % N
            mem_trace = memory.reconstruction_head(memory.state[char_id].unsqueeze(0))
            score = F.cosine_similarity(mem_trace, b_emb).item()
            pred = "consistent" if score >= best_thresh else "contradict"
            final_ids.append(row['id'])
            final_preds.append(pred)
    submission_df = pd.DataFrame({'id': final_ids, 'prediction': final_preds})
    submission_df.to_csv("results.csv", index=False)
    print(f"Saved {len(submission_df)} predictions to results.csv")
    print(submission_df.head())

MODEL_SAVE_PATH = "best_bdh_model.pth"
encoder = BDH_GPU().to(DEVICE)
mem_module = CausalMemory().to(DEVICE)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(mem_module.parameters()), lr=1e-3)
train_df = pd.read_csv(train_path)
if os.path.exists(MODEL_SAVE_PATH):
    print(f"Found saved model at {MODEL_SAVE_PATH}. Loading...")
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
    encoder.load_state_dict(checkpoint['encoder'])
    mem_module.load_state_dict(checkpoint['memory'])
    print(f"Model loaded! (Previous Best F1: {checkpoint.get('best_f1', 0):.4f})")
else:
    print("No saved model found. Starting training...")
    train_model_with_save(novel_files, train_df, encoder, mem_module, optimizer, epochs=5)
test_file_path = find_file('test.csv')
if test_file_path:
    generate_submission(train_df, test_file_path, encoder, mem_module)
else:
    print("Test file not found. Skipping prediction.")